## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

## Demo numpy based auto differentiation

In [2]:
import numpy as np

class Matmul:
    def __init__(self):
        self.mem = {}
        
    def forward(self, x, W):
        h = np.matmul(x, W)
        self.mem={'x': x, 'W':W}
        return h
    
    def backward(self, grad_y):
        '''
        x: shape(N, d)
        w: shape(d, d')
        grad_y: shape(N, d')
        '''
        x = self.mem['x']
        W = self.mem['W']
        
        ####################
        '''计算矩阵乘法的对应的梯度'''
        # y = x @ W
        # dL/dx = dL/dy @ W^T
        # dL/dW = x^T @ dL/dy
        grad_x = np.matmul(grad_y, W.T)
        grad_W = np.matmul(x.T, grad_y)
        ####################
        return grad_x, grad_W


class Relu:
    def __init__(self):
        self.mem = {}
        
    def forward(self, x):
        self.mem['x']=x
        return np.where(x > 0, x, np.zeros_like(x))
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        ####################
        '''计算relu 激活函数对应的梯度'''
        x = self.mem['x']
        grad_x = np.where(x > 0, grad_y, np.zeros_like(grad_y))
        ####################
        return grad_x
    


class Softmax:
    '''
    softmax over last dimention
    '''
    def __init__(self):
        self.epsilon = 1e-12
        self.mem = {}
        
    def forward(self, x):
        '''
        x: shape(N, c)
        '''
        x_exp = np.exp(x)
        partition = np.sum(x_exp, axis=1, keepdims=True)
        out = x_exp/(partition+self.epsilon)
        
        self.mem['out'] = out
        self.mem['x_exp'] = x_exp
        return out
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        s = self.mem['out']
        sisj = np.matmul(np.expand_dims(s,axis=2), np.expand_dims(s, axis=1)) # (N, c, c)
        g_y_exp = np.expand_dims(grad_y, axis=1)
        tmp = np.matmul(g_y_exp, sisj) #(N, 1, c)
        tmp = np.squeeze(tmp, axis=1)
        tmp = -tmp+grad_y*s 
        return tmp
    
class Log:
    '''
    softmax over last dimention
    '''
    def __init__(self):
        self.epsilon = 1e-12
        self.mem = {}
        
    def forward(self, x):
        '''
        x: shape(N, c)
        '''
        out = np.log(x+self.epsilon)
        
        self.mem['x'] = x
        return out
    
    def backward(self, grad_y):
        '''
        grad_y: same shape as x
        '''
        x = self.mem['x']
        
        return 1./(x+1e-12) * grad_y
    


## Gradient check

In [3]:
import tensorflow as tf

x = np.random.normal(size=[5, 6])
W = np.random.normal(size=[6, 4])
aa = Matmul()
out = aa.forward(x, W) # shape(5, 4)
grad = aa.backward(np.ones_like(out))
print (grad)

with tf.GradientTape() as tape:
    x, W = tf.constant(x), tf.constant(W)
    tape.watch(x)
    y = tf.matmul(x, W)
    loss = tf.reduce_sum(y)
    grads = tape.gradient(loss, x)
    print (grads)

import tensorflow as tf

x = np.random.normal(size=[5, 6])
aa = Relu()
out = aa.forward(x) # shape(5, 4)
grad = aa.backward(np.ones_like(out))
print (grad)

with tf.GradientTape() as tape:
    x= tf.constant(x)
    tape.watch(x)
    y = tf.nn.relu(x)
    loss = tf.reduce_sum(y)
    grads = tape.gradient(loss, x)
    print (grads)

import tensorflow as tf
x = np.random.normal(size=[5, 6], scale=5.0, loc=1)
label = np.zeros_like(x)
label[0, 1]=1.
label[1, 0]=1
label[1, 1]=1
label[2, 3]=1
label[3, 5]=1
label[4, 0]=1
print(label)
aa = Softmax()
out = aa.forward(x) # shape(5, 6)
grad = aa.backward(label)
print (grad)

with tf.GradientTape() as tape:
    x= tf.constant(x)
    tape.watch(x)
    y = tf.nn.softmax(x)
    loss = tf.reduce_sum(y*label)
    grads = tape.gradient(loss, x)
    print (grads)

import tensorflow as tf

x = np.random.normal(size=[5, 6])
aa = Log()
out = aa.forward(x) # shape(5, 4)
grad = aa.backward(label)
print (grad)

with tf.GradientTape() as tape:
    x= tf.constant(x)
    tape.watch(x)
    y = tf.math.log(x)
    loss = tf.reduce_sum(y*label)
    grads = tape.gradient(loss, x)
    print (grads)

(array([[-1.23977756, -3.61617998,  1.59669114,  2.53848233, -2.53991274,
        -0.65014603],
       [-1.23977756, -3.61617998,  1.59669114,  2.53848233, -2.53991274,
        -0.65014603],
       [-1.23977756, -3.61617998,  1.59669114,  2.53848233, -2.53991274,
        -0.65014603],
       [-1.23977756, -3.61617998,  1.59669114,  2.53848233, -2.53991274,
        -0.65014603],
       [-1.23977756, -3.61617998,  1.59669114,  2.53848233, -2.53991274,
        -0.65014603]]), array([[ 0.45555112,  0.45555112,  0.45555112,  0.45555112],
       [-3.35409056, -3.35409056, -3.35409056, -3.35409056],
       [-0.60083745, -0.60083745, -0.60083745, -0.60083745],
       [ 0.35199439,  0.35199439,  0.35199439,  0.35199439],
       [ 0.89590568,  0.89590568,  0.89590568,  0.89590568],
       [-0.02874226, -0.02874226, -0.02874226, -0.02874226]]))
Tensor("MatMul_1:0", shape=(5, 6), dtype=float64)
[[1. 1. 0. 0. 1. 0.]
 [1. 1. 0. 1. 0. 1.]
 [0. 1. 1. 1. 0. 1.]
 [0. 0. 0. 1. 0. 1.]
 [0. 1. 0. 0. 0. 1.]

d:\anaconda1\envs\tf1_env\lib\site-packages\ipykernel_launcher.py:97: RuntimeWarning: invalid value encountered in log


# Final Gradient Check

In [4]:
import numpy as np
import tensorflow as tf

# --- 1) 只用 numpy 生成数据（不要再用 x = tf.constant(x) 覆盖 numpy 的 x）---
x_np = np.random.normal(size=(5, 6)).astype(np.float64)
W1_np = np.random.normal(size=(6, 5)).astype(np.float64)
W2_np = np.random.normal(size=(5, 6)).astype(np.float64)

label_np = np.zeros_like(x_np)
label_np[0, 1] = 1.
label_np[1, 0] = 1.
label_np[2, 3] = 1.
label_np[3, 5] = 1.
label_np[4, 0] = 1.

# 确保前面的类定义 cell 已执行
if any(name not in globals() for name in ["Matmul", "Relu", "Softmax", "Log"]):
    raise RuntimeError("请先运行上面定义 Matmul/Relu/Softmax/Log 的代码单元，再运行本单元。")

# --- 2) numpy 前向/反向 ---
mul_h1 = Matmul()
mul_h2 = Matmul()
relu = Relu()
softmax = Softmax()
log = Log()

h1 = mul_h1.forward(x_np, W1_np)
h1_relu = relu.forward(h1)
h2 = mul_h2.forward(h1_relu, W2_np)
prob_np = softmax.forward(h2)
log_prob_np = log.forward(prob_np)

dL_dprob_np = log.backward(label_np)  # dL/dprob
print("numpy dL/dprob:")
print(dL_dprob_np)
print("--" * 20)

# --- 3) TensorFlow 对比：eager 用 GradientTape；graph 用 tf.gradients + Session ---
if tf.executing_eagerly():
    x_tf = tf.constant(x_np, dtype=tf.float64)
    W1_tf = tf.Variable(W1_np, dtype=tf.float64)
    W2_tf = tf.Variable(W2_np, dtype=tf.float64)
    label_tf = tf.constant(label_np, dtype=tf.float64)

    with tf.GradientTape() as tape:
        h1 = tf.matmul(x_tf, W1_tf)
        h1_relu = tf.nn.relu(h1)
        h2 = tf.matmul(h1_relu, W2_tf)
        prob = tf.nn.softmax(h2, axis=1)
        tape.watch(prob)
        loss = tf.reduce_sum(label_tf * tf.math.log(prob))

    grad_prob = tape.gradient(loss, prob)
    print("tf dL/dprob (eager):")
    print(grad_prob.numpy())
else:
    tf.compat.v1.reset_default_graph()
    g = tf.Graph()
    with g.as_default():
        x_tf = tf.constant(x_np, dtype=tf.float64)
        W1_tf = tf.constant(W1_np, dtype=tf.float64)
        W2_tf = tf.constant(W2_np, dtype=tf.float64)
        label_tf = tf.constant(label_np, dtype=tf.float64)

        h1 = tf.matmul(x_tf, W1_tf)
        h1_relu = tf.nn.relu(h1)
        h2 = tf.matmul(h1_relu, W2_tf)
        prob = tf.nn.softmax(h2, axis=1)
        loss = tf.reduce_sum(label_tf * tf.math.log(prob))

        grad_prob = tf.compat.v1.gradients(loss, prob)[0]

        with tf.compat.v1.Session() as sess:
            grad_prob_val = sess.run(grad_prob)
            print("tf dL/dprob (graph):")
            print(grad_prob_val)

numpy dL/dprob:
[[0.00000000e+00 1.39463665e+02 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.22067233e+04 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 2.33360214e+01
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 1.29686748e+00]
 [1.56385802e+01 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]
----------------------------------------
tf dL/dprob (graph):
[[0.00000000e+00 1.39463665e+02 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.22067235e+04 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 2.33360214e+01
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 1.29686748e+00]
 [1.56385802e+01 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.

## 建立模型

In [5]:
class myModel:
    def __init__(self):
        
        self.W1 = np.random.normal(size=[28*28+1, 100])
        self.W2 = np.random.normal(size=[100, 10])
        
        self.mul_h1 = Matmul()
        self.mul_h2 = Matmul()
        self.relu = Relu()
        self.softmax = Softmax()
        self.log = Log()
        
        
    def forward(self, x):
        x = x.reshape(-1, 28*28)
        bias = np.ones(shape=[x.shape[0], 1])
        x = np.concatenate([x, bias], axis=1)
        
        self.h1 = self.mul_h1.forward(x, self.W1) # shape(5, 4)
        self.h1_relu = self.relu.forward(self.h1)
        self.h2 = self.mul_h2.forward(self.h1_relu, self.W2)
        self.h2_soft = self.softmax.forward(self.h2)
        self.h2_log = self.log.forward(self.h2_soft)
            
    def backward(self, label):
        self.h2_log_grad = self.log.backward(-label)
        self.h2_soft_grad = self.softmax.backward(self.h2_log_grad)
        self.h2_grad, self.W2_grad = self.mul_h2.backward(self.h2_soft_grad)
        self.h1_relu_grad = self.relu.backward(self.h2_grad)
        self.h1_grad, self.W1_grad = self.mul_h1.backward(self.h1_relu_grad)
        
model = myModel()


## 计算 loss

In [6]:
def compute_loss(log_prob, labels):
     return np.mean(np.sum(-log_prob*labels, axis=1))
    

def compute_accuracy(log_prob, labels):
    predictions = np.argmax(log_prob, axis=1)
    truth = np.argmax(labels, axis=1)
    return np.mean(predictions==truth)

def train_one_step(model, x, y):
    model.forward(x)
    model.backward(y)
    model.W1 -= 1e-5* model.W1_grad
    model.W2 -= 1e-5* model.W2_grad
    loss = compute_loss(model.h2_log, y)
    accuracy = compute_accuracy(model.h2_log, y)
    return loss, accuracy

def test(model, x, y):
    model.forward(x)
    loss = compute_loss(model.h2_log, y)
    accuracy = compute_accuracy(model.h2_log, y)
    return loss, accuracy

## 实际训练

In [8]:
train_data, test_data = mnist_dataset()
train_label = np.zeros(shape=[train_data[0].shape[0], 10])
test_label = np.zeros(shape=[test_data[0].shape[0], 10])
train_label[np.arange(train_data[0].shape[0]), np.array(train_data[1])] = 1.
test_label[np.arange(test_data[0].shape[0]), np.array(test_data[1])] = 1.

for epoch in range(100):
    loss, accuracy = train_one_step(model, train_data[0], train_label)
    print('epoch', epoch, ': loss', loss, '; accuracy', accuracy)
loss, accuracy = test(model, test_data[0], test_label)

print('test loss', loss, '; accuracy', accuracy)

epoch 0 : loss 6.655560049932523 ; accuracy 0.7011666666666667
epoch 1 : loss 6.958603075134358 ; accuracy 0.6949833333333333
epoch 2 : loss 6.3873513913234214 ; accuracy 0.7130166666666666
epoch 3 : loss 6.533692863211798 ; accuracy 0.7102666666666667
epoch 4 : loss 6.4713674673069805 ; accuracy 0.7091333333333333
epoch 5 : loss 6.961274016600109 ; accuracy 0.6963166666666667
epoch 6 : loss 6.080168938185173 ; accuracy 0.7278166666666667
epoch 7 : loss 6.016412904044972 ; accuracy 0.73045
epoch 8 : loss 6.1004976080628435 ; accuracy 0.7262833333333333
epoch 9 : loss 6.338161632041869 ; accuracy 0.7183166666666667
epoch 10 : loss 6.14293180085913 ; accuracy 0.7249333333333333
epoch 11 : loss 6.3125251288960715 ; accuracy 0.71965
epoch 12 : loss 5.986707843744444 ; accuracy 0.7324666666666667
epoch 13 : loss 6.0355183151925464 ; accuracy 0.7302666666666666
epoch 14 : loss 5.844354955582449 ; accuracy 0.7390833333333333
epoch 15 : loss 5.862975655159635 ; accuracy 0.7375333333333334
epoc